Ce code permet de créer le panel final qui nous servira pour les estimations.
À partir des fichiers panel_va_STAN.csv & panel_cit1.csv, on procède à plusieurs modifications :
 - création des secteurs agrégés C & H
 - création du vecteur d'intensité de R&D
 - création des agrégats de R&D : intra-sectoriel, extra-sectoriel, domestique et étranger
 - ajout d'une DUMMY pour la crise de 2009
 - calcul de l'indicateur HHI (Herfindahl-Hirschmann) par secteur
 - séparation en 2 panels distincts selon l'intensité de R&D moyenne pour chaque pays, pour chaque base de données

Ce code retourne des panels longs pour chaque base prêt à être utilisé pour les estimations.

Version 2.0 : lit le panel du run choisi dans `project_settings.py` (RD_UNIT, DIMENSION, TECH), lit l'intensité `intensite_propre.csv` (RDV / PRODV, `prépa_va_STAN`) et écrit les panels dans `runs/<run>/panels/`.

In [16]:
import numpy as np
import pandas as pd
import importlib
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, METRIC_WANTED, METRIC_SPE, YEARS, RESULTS_DIR, RUN_DIR, PANEL_CODE_DIR, CHOC, CHOC_SCENARIO
print(f"CHOC = {CHOC}" + (f" (scenario {CHOC_SCENARIO})" if CHOC else "") + f" -> {RESULTS_DIR}")
import os

CHOC = False -> D:/ENTICE/runs/SRDQ (c,s)/


In [17]:
panel_file = RESULTS_DIR + f"panel_estimation/panel_estimation_{BASE}.csv"
print("Panel amont lu :", panel_file)
df = pd.read_csv(panel_file)

# La nouvelle mesure doit déjà être présente dans le CSV produit par calc_blocs.
if BASE == "CAPITAL" and METRIC_WANTED == "embod_investment":
    investment_metric = METRIC_SPE["CAPITAL_embod_investment"]
    investment_columns = [
        f"rd_same_sector_same_country_{investment_metric}",
        f"rd_same_sector_diff_country_{investment_metric}",
        f"rd_diff_sector_same_country_{investment_metric}",
        f"rd_diff_sector_diff_country_{investment_metric}",
    ]
    missing = [c for c in investment_columns if c not in df.columns]
    if missing:
        raise ValueError(
            f"Colonnes investment absentes du panel lu : {missing}. "
            "Relancer calc_blocs modifié avec BASE='CAPITAL' et "
            "METRIC_WANTED='embod_investment', en vérifiant le même RESULTS_DIR. "
            f"Fichier attendu : {panel_file}"
        )


Panel amont lu : D:/ENTICE/runs/SRDQ (c,s)/panel_estimation/panel_estimation_FDI.csv


In [18]:
print(df.columns)

Index(['Region', 'Sector', 'year', 'rd_same_sector_same_country_rd_dir_fdips',
       'rd_same_sector_diff_country_rd_dir_fdips',
       'rd_diff_sector_same_country_rd_dir_fdips',
       'rd_diff_sector_diff_country_rd_dir_fdips'],
      dtype='str')


In [19]:
df = df.rename(columns={"year": "annee"})

Création des secteurs agrégés

In [ ]:
c_subsectors = df.loc[df['Sector'].str.startswith('C', na=False), 'Sector'].unique().tolist()

mask_c = df['Sector'].isin(c_subsectors)

phys = ["FIGARO", "EXIOBASE", "GTAP", "OECD_ICIO", "CAPITAL"]

if BASE in phys:
    col = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',

            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',

            'exports',
            'imports_final_demand',
            'imports_intermediate',
            'production']
else:
    col = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}']

if BASE == "CAPITAL" and "rd_same_sector_same_country_rd_embod_investment_CAPITAL" in df.columns:
    col += ["rd_same_sector_same_country_rd_embod_investment_CAPITAL",
            "rd_same_sector_diff_country_rd_embod_investment_CAPITAL",
            "rd_diff_sector_same_country_rd_embod_investment_CAPITAL",
            "rd_diff_sector_diff_country_rd_embod_investment_CAPITAL",]

c_agg = (
    df[mask_c]
    .groupby(['Region', 'annee'], as_index=False)
    [col]
    .sum()
)
c_agg['Sector'] = 'C'

df_final = pd.concat([df, c_agg], ignore_index=True)

In [6]:
if BASE in phys:
    h_subsectors = df.loc[df['Sector'].str.startswith('H', na=False), 'Sector'].unique().tolist()
    mask_h = df_final['Sector'].isin(h_subsectors)

    h_agg = (
        df_final[mask_h]
        .groupby(['Region', 'annee'], as_index=False)
        [col]
        .sum()
    )
    h_agg['Sector'] = 'H'

    df_final1 = pd.concat([df_final, h_agg], ignore_index=True)
else:
    df_final1 = df_final

In [7]:
os.makedirs(RESULTS_DIR + 'panels', exist_ok=True)
df_final1.to_csv(RESULTS_DIR + f'panels/panel_{BASE}_intermediaire_F.csv')

In [8]:
df_final1

,Region,Sector,annee,rd_same_sector_same_country_rd_dir_CI,rd_same_sector_diff_country_rd_dir_CI,rd_diff_sector_same_country_rd_dir_CI,rd_diff_sector_diff_country_rd_dir_CI,rd_same_sector_same_country_rd_embod_CI,rd_same_sector_diff_country_rd_embod_CI,rd_diff_sector_same_country_rd_embod_CI,rd_diff_sector_diff_country_rd_embod_CI,exports,imports_final_demand,imports_intermediate,production
0,AT,A,1995,0.804175,0.952200,40.306354,19.813212,1.060669,2.531867,76.269374,118.469685,718.475369,3.598103e+04,492.385575,9.172724e+03
1,AT,B,1995,3.542874,0.915788,9.070018,7.741217,4.398917,2.579031,19.621463,36.111032,275.880215,3.598103e+04,165.452501,2.023289e+03
2,AT,C10_C12,1995,9.530204,7.166005,92.053679,87.233782,11.903051,12.963555,192.546512,392.651669,5552.072022,3.598103e+04,2795.282794,1.772503e+04
3,AT,C13_C15,1995,13.688819,15.845573,32.848746,84.132103,15.724666,24.047869,56.362840,261.832631,4709.394247,3.598103e+04,2050.518770,6.068430e+03
4,AT,C16_C18,1995,35.933283,18.990563,70.868978,75.775115,45.006637,32.223518,158.749000,414.931639,8667.458120,3.598103e+04,3394.563410,1.717312e+04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45355,US,H,2018,51227.016836,1338.076283,41691.378920,4746.633994,63135.420813,2076.774063,79755.812081,19053.378032,145814.954979,7.113276e+06,101633.695669,2.563330e+06
45356,US,H,2019,57708.642191,1583.145270,46661.573520,5213.467037,71104.957435,2450.752748,87900.194892,20959.245466,153658.047089,8.123278e+06,115234.674594,2.833351e+06
45357,US,H,2020,64369.361010,1736.908595,49849.158664,5406.077093,79309.458770,2689.049949,93334.628991,21625.502964,153658.047089,8.123278e+06,115234.674594,2.833351e+06
45358,US,H,2021,66079.181163,1954.156458,52670.905581,5647.352872,81413.702031,3023.349747,98307.133203,22469.022269,153658.047089,8.123278e+06,115234.674594,2.833351e+06


Découpe des différents panels

In [9]:
if BASE in phys:
    va_STAN = pd.read_csv(PATH_WRK + 'code/création_panel_estimation/Panels_intermediaires/panel_va_STAN.csv')
    va_STAN = va_STAN[va_STAN["annee"].isin(YEARS)]
    va_STAN = va_STAN[va_STAN["Sector"].isin(df_final1["Sector"])]

    df_final1 = df_final1.rename(columns={"production": f"production_{BASE}"})
    df = va_STAN.merge(df_final1, how='left', on=['Region', 'Sector', 'annee'])
    df = df.drop(columns=["Unnamed: 0"])
else:
    va_STAN = pd.read_csv(PATH_WRK + 'code/création_panel_estimation/Panels_intermediaires/panel_cit1.csv')
    va_STAN = va_STAN[va_STAN["annee"].isin(YEARS)]
    va_STAN = va_STAN[va_STAN["Sector"].isin(df_final1["Sector"])]
    df = va_STAN.merge(df_final1, how='left', on=['Region', 'Sector', 'annee'])
    df = df.drop(columns=["Unnamed: 0"])

df

,Region,Sector,annee,VA_V,VA_Q,VA_costFactors,comp_emp,emp,net_stock_capital,production,...,rd_diff_sector_same_country_rd_dir_CI,rd_diff_sector_diff_country_rd_dir_CI,rd_same_sector_same_country_rd_embod_CI,rd_same_sector_diff_country_rd_embod_CI,rd_diff_sector_same_country_rd_embod_CI,rd_diff_sector_diff_country_rd_embod_CI,exports,imports_final_demand,imports_intermediate,production_GTAP
0,AT,A,1995,3843.823732,3410.753099,5023.693732,348.500000,38.626784,3.428939e+04,7664.729833,...,40.306354,19.813212,1.060669,2.531867,76.269374,118.469685,718.475369,3.598103e+04,492.385575,9172.723729
1,AT,A,1996,3588.173927,3354.544562,4309.291927,354.200000,38.047075,3.438198e+04,7436.299892,...,42.724423,20.078674,1.157559,2.515352,80.761892,119.809972,718.475369,3.598103e+04,492.385575,9172.723729
2,AT,A,1997,3693.656888,3478.446066,4251.222888,364.700000,38.474740,3.479585e+04,7714.325937,...,45.330475,20.378901,1.256571,2.460165,85.617174,121.342921,718.475369,3.598103e+04,492.385575,9172.723729
3,AT,A,1998,3590.057497,3625.242586,4249.225497,366.000000,37.816498,3.498068e+04,7467.688729,...,48.516325,20.632274,1.370090,2.394955,91.571450,122.003931,718.475369,3.598103e+04,492.385575,9172.723729
4,AT,A,1999,3616.638040,3843.736884,4315.897040,382.300000,39.327879,3.516822e+04,7480.581422,...,52.616317,21.002000,1.513661,2.348427,99.475057,123.213370,718.475369,3.598103e+04,492.385575,9172.723729
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15199,US,R+S+T,2015,403824.172649,433197.497956,393560.340571,267416.971634,10318.000000,1.022369e+06,670507.180879,...,9450.445496,497.246960,30.395149,1.029124,17620.674702,2337.382359,9174.073716,1.301927e+06,13449.065198,739747.264946
15200,US,R+S+T,2016,420530.767816,438314.944498,409734.759237,278968.564174,10471.000000,1.047762e+06,701452.164244,...,10141.935713,573.153101,33.105778,1.163923,19037.960384,2639.935094,8288.217628,1.422655e+06,15068.993539,845105.690845
15201,US,R+S+T,2017,435249.731220,449286.366872,423538.087025,289233.014358,10519.000000,1.062726e+06,727146.899842,...,10765.319306,600.089502,36.501267,1.271818,20150.862493,2756.202252,8288.217628,1.422655e+06,15068.993539,845105.690845
15202,US,R+S+T,2018,457138.704255,465631.493211,444723.251620,304570.694274,10767.000000,1.082054e+06,763918.279636,...,11452.728574,650.972490,39.746994,1.333862,21346.514923,2903.778411,8288.217628,1.422655e+06,15068.993539,845105.690845


In [10]:
# embod
if BASE in phys:
    col_F = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            ]
    col_D = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod_F'] = df[col_F].sum(axis=1)
    df['rd_embod_D'] = df[col_D].sum(axis=1)

    col_I = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            ]
    col_X = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod_I'] = df[col_I].sum(axis=1)
    df['rd_embod_X'] = df[col_X].sum(axis=1)

    col_S = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod'] = df[col_S].sum(axis=1)

#dir
col_F = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        ]
col_D = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir_F'] = df[col_F].sum(axis=1)
df['rd_dir_D'] = df[col_D].sum(axis=1)

col_I = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        ]
col_X = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir_I'] = df[col_I].sum(axis=1)
df['rd_dir_X'] = df[col_X].sum(axis=1)

col_S = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir'] = df[col_S].sum(axis=1)

df["2009_crisis"]=(df["annee"]==2009).astype(int)

# col = 'production_EXIOBASE_mapF'
# df['share'] = df['production'] / df.groupby(['Sector', 'annee'])[col].transform('sum')
df['share'] = df['production'] / df.groupby(['Sector', 'annee'])['production'].transform('sum')
df['HHI'] = df.groupby(['Sector', 'annee'])['share'].transform(lambda s: (s**2).sum())

if BASE == "CAPITAL" and "rd_same_sector_same_country_rd_embod_investment_CAPITAL" in df.columns:
    same_same = "rd_same_sector_same_country_rd_embod_investment_CAPITAL"
    same_diff = "rd_same_sector_diff_country_rd_embod_investment_CAPITAL"
    diff_same = "rd_diff_sector_same_country_rd_embod_investment_CAPITAL"
    diff_diff = "rd_diff_sector_diff_country_rd_embod_investment_CAPITAL"

    df["rd_embod_investment_D"] = df[[same_same, diff_same]].sum(axis=1)
    df["rd_embod_investment_F"] = df[[same_diff, diff_diff]].sum(axis=1)
    df["rd_embod_investment_I"] = df[[same_same, same_diff]].sum(axis=1)
    df["rd_embod_investment_X"] = df[[diff_same, diff_diff]].sum(axis=1)
    df["rd_embod_investment"] = df[[same_same, same_diff, diff_same, diff_diff]].sum(axis=1)

In [11]:
df = df.drop(columns=['share'])

On ajoute l'intensité en R&D

In [12]:
df['intensité'] = df[f'rd_{METRIC_WANTED}'] / df[f'production']

In [13]:
df.to_csv(RESULTS_DIR + f'panels/panel_{BASE}_F.csv', index=False)

Division des panels par base en deux groupes de pays en fonction de leur intensité de R&D moyenne

In [14]:
# R&D propre / production : fichier RDV / PRODV calculé une seule fois par prépa_va_STAN (indépendant de la base et de la technologie)
# rd_propre = flux de R&D en M€ courants, production = production STAN aux prix courants en M€ (taux de change de l'année), C et H inclus
df_int = (pd.read_csv(PANEL_CODE_DIR + "Panels_intermediaires/intensite_propre.csv")
            .rename(columns={"rd_propre_V": "rd_propre", "production_V": "production"}))

# vraie intensité = R&D propre du secteur / production du secteur
df_int["intensité_vraie"] = df_int["rd_propre"] / df_int["production"]

# intensité annuelle propre R/Y (en points de %), pour le terme croisé de l'estimation
df_int.loc[df_int["production"] <= 1.0, "intensité_vraie"] = np.nan   # production quasi nulle => intensité absurde
int_annual = (df_int[["Region", "Sector", "annee", "intensité_vraie"]]
              .rename(columns={"intensité_vraie": "int_propre"}))
int_annual["int_propre"] = 100 * int_annual["int_propre"]


In [15]:
# Panel complet, avec int_propre
df = df.merge(int_annual, on=["Region", "Sector", "annee"], how="left")

os.makedirs(RESULTS_DIR + "panels", exist_ok=True)

df.to_csv(RESULTS_DIR + f"panels/panel_{BASE}_F.csv", index=False)
